# SH-compression Colab runner
Thin execution shell — all logic lives in the repository (`src/`, `scripts/`).
Runs top-to-bottom on a fresh GPU runtime (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Parameters
REPO_URL = 'https://github.com/adash963/3DGS.git'
BRANCH = 'dissertation'
COMMIT = ''  # optional: pin an exact commit; empty = branch tip
STAGE = 'all'  # all (full reproduction) | stage1 | stage2 | stage4 | stage5 | stage7


In [ ]:
# 1. Environment diagnostics
import json, os, platform, subprocess, sys, torch
env = {
  'python': sys.version,
  'platform': platform.platform(),
  'torch': torch.__version__,
  'cuda_available': torch.cuda.is_available(),
  'cuda_version': torch.version.cuda,
  'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/2**30, 1) if torch.cuda.is_available() else None,
}
print(json.dumps(env, indent=2))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'


In [ ]:
# 2. Clone repo at pinned ref
%cd /content
!rm -rf 3DGS
!git clone --recursive -b $BRANCH $REPO_URL
%cd /content/3DGS
if COMMIT:
    !git checkout $COMMIT && git submodule update --init --recursive
!git rev-parse HEAD | tee /content/git_commit.txt


In [ ]:
# 3. Install deps + compile CUDA rasteriser (logs saved)
!pip install -q -r requirements-colab.txt 2>&1 | tee /content/install.log
!pip install -q -e . 2>&1 | tee -a /content/install.log
!pip install -q ./submodules/diff-gaussian-rasterization 2>&1 | tee /content/compile.log
!pip install -q ./submodules/simple-knn 2>&1 | tee -a /content/compile.log
import importlib
import diff_gaussian_rasterization, simple_knn
print('rasteriser + knn import OK')


In [ ]:
# 4. Data: pretrained model (small) + dataset (cameras/images)
%mkdir -p /content/work
%cd /content/work
![ -f GaussianViewTest.zip ] || wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/GaussianViewTest.zip
![ -d GaussianViewTest ] || unzip -q GaussianViewTest.zip
![ -f tandt_db.zip ] || wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
![ -d tandt ] || unzip -q tandt_db.zip
!ls GaussianViewTest/model/point_cloud/
%cd /content/3DGS
# checkpoint ships a Windows-authored cfg_args that crashes render.py's eval();
# rewrite it with POSIX paths and eval=True (test split = every 8th camera)
!python scripts/fix_cfg_args.py --model-dir /content/work/GaussianViewTest/model --source-path /content/work/tandt/train


In [ ]:
# 5. CPU unit tests + model inspection (smoke)
!python -m pytest tests/unit -q
!python scripts/inspect_model.py --ply /content/work/GaussianViewTest/model/point_cloud/iteration_30000/point_cloud.ply


In [ ]:
# 6. Stage command
import json, subprocess, sys
MODEL = '/content/work/GaussianViewTest/model'
PLY = f'{MODEL}/point_cloud/iteration_30000/point_cloud.ply'
SOURCE = '/content/work/tandt/train'
BASE_RENDERS = f'{MODEL}/test/ours_30000/renders'
GT_DIR = f'{MODEL}/test/ours_30000/gt'

def run(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f'FAILED: {cmd}'

if STAGE in ('all', 'stage1'):
    # baseline render of the eval set with the unmodified renderer
    run(f'python render.py -m {MODEL} -s {SOURCE} --skip_train')
    run(f'ls {BASE_RENDERS} | wc -l')
if STAGE in ('all', 'stage2'):
    run(f'python -c "import sys; sys.path.insert(0,\'src\'); from sh_compression.extraction import roundtrip_unchanged; import json; r=roundtrip_unchanged(\'{PLY}\', \'/content/work/roundtrip.ply\'); print(json.dumps(r)); assert r[\'max_abs_coeff_diff\']==0.0"')
if STAGE == 'all':
    # reproduce the headline configuration end-to-end and check its metrics
    run('python scripts/run_render_eval.py --config configs/quantization_ablation.yaml'
        f' --names sq_r16_b8 --ply {PLY} --source {SOURCE}'
        f' --baseline-renders {BASE_RENDERS} --gt-dir {GT_DIR}'
        ' --out-root /content/work/repro --no-lpips')
    m = json.load(open('/content/work/repro/image_metrics/metrics_sq_r16_b8.json'))
    s = m['summary']
    print(json.dumps(s, indent=2))
    assert s['vsbase_psnr'] > 34.0 and s['gt_psnr'] > 24.0 and s['vsbase_ssim'] > 0.99, s
    assert m['asset_file_bytes'] < 15_000_000, m['asset_file_bytes']
if STAGE == 'stage4':
    run(f'python scripts/analyse_rank.py --ply {PLY}')
    run(f'python scripts/compress_model.py --ply {PLY} --rank 8 --mode lowrank --out /content/work/assets/lr8.npz')
if STAGE == 'stage5':
    run(f'python scripts/compress_model.py --ply {PLY} --rank 8 --mode vq --codebook-size 256 --out /content/work/assets/vq8_256.npz')
if STAGE == 'stage7':
    run(f'python scripts/run_ablation.py --config configs/quantization_ablation.yaml --ply {PLY} --out-dir results/experiments/ablation')
print('STAGE', STAGE, 'PASS')
